# MIND article embeddings

MIND ships no article vectors, so they are generated once here and downloaded as an
artifact rather than recomputed on every machine — the dev machine has integrated
graphics only. EB-NeRD needs none of this: it ships precomputed multilingual BERT
vectors, which `pipeline/embed.py` reads directly.

**Runtime → Change runtime type → T4 GPU** before running anything.

Everything specific to MIND — the model, the vector width, the token window, the file
names — is read from the dataset registry rather than written here, so this notebook
cannot drift from what the pipeline expects to load. Run it top to bottom; the last
cell says what to do with the output.

## 1. The repository

Cloned rather than uploaded, so this notebook encodes the same text the pipeline does:
the corpus comes from the repo's own ingest, and both the text and the encoding come
from `pipeline/embed.py`.

**Nothing is installed.** Colab already ships everything this needs — `torch` and
`transformers` to encode, `pandas`, `numpy`, `requests` and `tqdm` for the pipeline
modules imported below. That is deliberate rather than lucky: installing anything here
risks pip re-pinning `numpy` underneath a kernel that has already imported it, which
leaves numpy half-replaced on disk and kills every later import with
`cannot import name '_center' from 'numpy._core.umath'`. With no install there is
nothing to go wrong.

`embed.encode` is the reason this is possible: it does what the `sentence-transformers`
wrapper does for this checkpoint — a 6-layer BERT, mean pooling over the real tokens,
L2 normalisation — against `transformers` directly. The vectors are the same ones, to
within 9.3e-08 on real MIND text.

(`requirements.txt` is still the right file for running the *whole* pipeline on a hosted
runtime, which does need `bm25s`, `nltk` and `faiss-cpu`. This notebook is a narrower
job, and installing that file is what broke it before.)

In [ ]:
![ -d vub-news ] || git clone --depth 1 https://github.com/vubsss/vub-news.git
%cd /content/vub-news

## 2. Check the runtime

The GPU is asserted here rather than at encode time, so a CPU runtime is caught before
a 100 MB download rather than after it. If any of these imports fail, the runtime is in
a state this notebook did not create — **Runtime → Restart session** and start again.

In [ ]:
import numpy as np
import pandas as pd
import torch
import transformers

assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> T4 GPU'
print(f'numpy {np.__version__}, pandas {pd.__version__}')
print(f'torch {torch.__version__}, transformers {transformers.__version__}')
print(torch.cuda.get_device_name(0))

## 3. A HuggingFace token

MIND is downloaded from the `yjw1029/MIND` mirror, because the official Microsoft
endpoint returns HTTP 409. That mirror is a **gated repo**, so a token alone is not
enough:

1. open <https://huggingface.co/datasets/yjw1029/MIND> while logged in and accept its
   terms — a token from an account that has not accepted them is still refused
2. create a read token at <https://huggingface.co/settings/tokens>

`acquire` reads the token from the environment, which is all a throwaway Colab clone
needs — the repo's `.env` file exists so the token survives across shells on a machine
you come back to. `getpass` keeps it out of the saved notebook.

In [ ]:
import os
from getpass import getpass

os.environ['HF_TOKEN'] = getpass('HF_TOKEN: ')

## 4. The article corpus

Only `acquire` and `ingest` are needed: embeddings come from title and abstract, and
`preprocess` builds `lexical_text`, which is BM25's input, not this one. `build.py` is
deliberately not imported — it would pull in the whole stage table, and with it `bm25s`,
`nltk` and `scikit-learn`, none of which Colab ships or this notebook uses.

In [ ]:
from pipeline import acquire, embed, ingest
from pipeline.datasets import DATASETS

MIND = DATASETS['mind']

acquire.run(MIND, False)
ingest.run(MIND, False)

In [ ]:
articles = pd.read_parquet(MIND.feature_store_dir / 'articles.parquet')
text = embed.document_text(articles)

print(f'{len(articles):,} articles')
print(text.head(3).to_string())

## 5. Encode

`embed.encode` normalises as it goes, which is what lets the registry declare
`normalise=False` for MIND: the pipeline verifies unit length on load rather than
redoing it. Ticket 8 indexes these with an inner product and calls the result a cosine
similarity, which is true only of unit vectors.

The elapsed time is worth noting down — ticket 16's ten-times-scale section is supposed
to cite measured figures.

In [ ]:
import time

started = time.perf_counter()
vectors = embed.encode(text.tolist(), MIND, device='cuda', batch_size=256)
elapsed = time.perf_counter() - started
print(f'encoded {len(vectors):,} articles in {elapsed:.1f} s ({len(vectors) / elapsed:,.0f}/s)')

### Check it before downloading it

Both of these are checked again by the pipeline when it loads the artifact. Checking
here means a bad run is caught on the GPU that produced it, rather than an hour later
on a machine that cannot regenerate it.

In [ ]:
assert vectors.shape == (len(articles), MIND.embeddings.dim), vectors.shape
embed.check_unit_norm(vectors, MIND)
print(f'{vectors.shape[0]:,} x {vectors.shape[1]}, unit length')

## 6. Write the artifact

Two files, and the order of the ids is load-bearing: `embed.read_source` pairs them
with the matrix positionally before `embed.align` reorders both onto whatever the
corpus turns out to hold. So the ids are written in **encode order**, which is the
order `articles` came in — not sorted, and not the corpus order of some later run.

In [ ]:
MIND.artifacts_dir.mkdir(parents=True, exist_ok=True)
np.save(MIND.artifacts_dir / MIND.embeddings.artifact, vectors)
pd.DataFrame({'article_id': articles['article_id']}).to_parquet(
    MIND.artifacts_dir / embed.ID_INDEX, index=False
)

!ls -la {MIND.artifacts_dir}

## 7. Upload to Drive

The pipeline fetches a Drive **folder**, so both files go in one folder together.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

folder = Path('/content/drive/MyDrive/vub-news-embeddings')
folder.mkdir(parents=True, exist_ok=True)
!cp {MIND.artifacts_dir}/{MIND.embeddings.artifact} {folder}/
!cp {MIND.artifacts_dir}/{embed.ID_INDEX} {folder}/
!ls -la {folder}

## 8. Then, once

1. In Drive, right-click **vub-news-embeddings** → Share → *Anyone with the link* →
   Viewer. `gdown` cannot read a private folder, and fails in a way that looks like a
   missing file rather than a permission problem.
2. Copy the folder id out of its URL — the part after `/folders/`:
   `https://drive.google.com/drive/folders/`**`1AbC...xyz`**
3. Set it in `pipeline/datasets.py`, on MIND's `EmbeddingSpec`:

   ```python
   gdrive_file_id="1AbC...xyz",
   ```
4. `python build.py --dataset mind` now downloads the artifact instead of stopping,
   and every machine after this one gets it without a GPU.

Re-run this notebook only if the article corpus changes or the registry names a
different model — the vectors are keyed by article id, so a corpus that gained articles
would leave them on zero rows, which `build.py` reports as a missing-vector count.